In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y monai monai-weekly >/dev/null 2>&1

!pip install -q \
    "monai>=1.4.0" \
    nibabel \
    h5py \
    pyyaml \
    tensorboardX \
    scikit-image \
    scipy \
    tqdm \
    wandb

import torch, monai
print("✓ PyTorch:", torch.__version__)
print("✓ CUDA:   ", torch.version.cuda)
print("✓ MONAI:  ", monai.__version__)

In [ ]:
!git clone https://github.com/cbmusonda/RCPS_ACDC.git
!git clone https://github.com/SongwuJob/CML.git

In [ ]:
import os

# CML repo location inside Colab
CML_BASE        = "/content/CML"
ACDC_TARGET_DIR = os.path.join(CML_BASE, "data", "ACDC")

# Your ACDC dataset in Drive (from your message)
DRIVE_ACDC_DIR = "/content/drive/Datasets/ACDC"

# Reset / create destination
!rm -rf "$ACDC_TARGET_DIR"
!mkdir -p "$ACDC_TARGET_DIR"

# Copy everything: lists + data folder
!cp -r "$DRIVE_ACDC_DIR"/* "$ACDC_TARGET_DIR"

print("Contents of ACDC_TARGET_DIR:")
!ls -R "$ACDC_TARGET_DIR"


In [ ]:
import os
import h5py
import nibabel as nib
import numpy as np

# ===== Base paths =====
CML_BASE       = "/content/CML"
ACDC_BASE      = os.path.join(CML_BASE, "data", "ACDC")
ACDC_RCPS_ROOT = os.path.join(CML_BASE, "data", "ACDC_rcps")

# HDF5 volume location:
#   /content/CML/data/ACDC/data/patientXXX_frameYY.h5
H5_ROOT = os.path.join(ACDC_BASE, "data")

TRAIN_LIST = os.path.join(ACDC_BASE, "train.list")
VAL_LIST   = os.path.join(ACDC_BASE, "val.list")

os.makedirs(os.path.join(ACDC_RCPS_ROOT, "train_images"), exist_ok=True)
os.makedirs(os.path.join(ACDC_RCPS_ROOT, "train_labels"), exist_ok=True)
os.makedirs(os.path.join(ACDC_RCPS_ROOT, "val_images"), exist_ok=True)
os.makedirs(os.path.join(ACDC_RCPS_ROOT, "val_labels"), exist_ok=True)

def convert_split(list_path, img_out_dir, lbl_out_dir):
    with open(list_path) as f:
        cases = [l.strip() for l in f if l.strip()]

    print(f"\nConverting {len(cases)} cases from {list_path} ...")
    missing = 0

    for case in cases:
        h5_path = os.path.join(H5_ROOT, case + ".h5")
        if not os.path.exists(h5_path):
            print(f"!! Missing H5 file for {case}: {h5_path}")
            missing += 1
            continue

        with h5py.File(h5_path, "r") as hf:
            if "image" not in hf or "label" not in hf:
                print(f"!! 'image'/'label' dataset missing in {h5_path}")
                missing += 1
                continue

            img = hf["image"][()]
            lbl = hf["label"][()]

        # If 2D (H, W), add a dummy Z axis
        if img.ndim == 2:
            img = img[None, ...]
            lbl = lbl[None, ...]

        img = img.astype(np.float32)
        lbl = lbl.astype(np.float32)

        img_nii = nib.Nifti1Image(img, np.eye(4))
        lbl_nii = nib.Nifti1Image(lbl, np.eye(4))

        nib.save(img_nii, os.path.join(img_out_dir, case + ".nii.gz"))
        nib.save(lbl_nii, os.path.join(lbl_out_dir, case + ".nii.gz"))

    print(f"Done. Missing {missing} cases")

# Build RCPS-style ACDC dataset
convert_split(
    TRAIN_LIST,
    os.path.join(ACDC_RCPS_ROOT, "train_images"),
    os.path.join(ACDC_RCPS_ROOT, "train_labels"),
)

convert_split(
    VAL_LIST,
    os.path.join(ACDC_RCPS_ROOT, "val_images"),
    os.path.join(ACDC_RCPS_ROOT, "val_labels"),
)

print("\nFinal ACDC_rcps structure:")
!ls -R "$ACDC_RCPS_ROOT"


In [ ]:
import os

# Create the directory structure RCPS expects
os.makedirs("/content/CML-main/CML-main/data", exist_ok=True)

# Create symbolic link to actual data location
if not os.path.exists("/content/CML-main/CML-main/data/ACDC_rcps"):
    os.symlink("/content/CML/data/ACDC_rcps", "/content/CML-main/CML-main/data/ACDC_rcps")
    print("✓ Data path fixed!")

In [ ]:
%cd /content/RCPS_ACDC

!python train.py \
    --task acdc \
    --exp_name acdc_colab_400epochs \
    --ncpu 2 \
    --mixed \
    --eval_interval 5 \
    --save_interval 10 \
    --verbose

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

# ====== PATHS ======
CML_BASE       = "/content/CML"
ACDC_RCPS_ROOT = os.path.join(CML_BASE, "data", "ACDC_rcps")

IMAGE_DIR = os.path.join(ACDC_RCPS_ROOT, "val_images")
LABEL_DIR = os.path.join(ACDC_RCPS_ROOT, "val_labels")

# 🔧 make sure this matches the folder name in inference_display/acdc/
EXP_NAME = "acdc_colab_400epochs-task_acdc-ratio_0.1"
PRED_DIR = f"/content/RCPS_ACDC/experiments/inference_display/acdc/{EXP_NAME}"

print("IMAGE_DIR:", IMAGE_DIR)
print("LABEL_DIR:", LABEL_DIR)
print("PRED_DIR :", PRED_DIR)

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith('_pred.nii.gz')])
print("Num prediction files:", len(pred_files))

# ====== VISUALIZATION LOGIC ======
def visualize_case(case_file):
    case_id = case_file.replace('_pred.nii.gz', '')
    img_path = os.path.join(IMAGE_DIR, case_id + ".nii.gz")
    lbl_path = os.path.join(LABEL_DIR, case_id + ".nii.gz")
    pred_path = os.path.join(PRED_DIR, case_file)

    print(f"\nCase: {case_id}")
    print("Image path:", img_path)
    print("Label path:", lbl_path)
    print("Pred  path:", pred_path)

    img = nib.load(img_path).get_fdata()
    lbl = nib.load(lbl_path).get_fdata()
    pred = nib.load(pred_path).get_fdata()

    # assume (Z, H, W). If not, you can tweak axis.
    depth = img.shape[0]

    @interact(slice_idx=IntSlider(min=0, max=depth-1, step=1, value=depth//2))
    def _show(slice_idx):
        fig, axes = plt.subplots(1, 3, figsize=(15,5))
        axes[0].imshow(img[slice_idx], cmap='gray')
        axes[0].set_title(f"Image (slice {slice_idx})")

        axes[1].imshow(lbl[slice_idx])
        axes[1].set_title("GT")

        axes[2].imshow(pred[slice_idx])
        axes[2].set_title("Prediction")

        for ax in axes:
            ax.axis('off')
        plt.show()

# ====== TOP-LEVEL DROPDOWN ======
case_dropdown = Dropdown(
    options=pred_files,
    description='Case:',
    value=pred_files[0] if pred_files else None,
    disabled=False,
)

interact(visualize_case, case_file=case_dropdown)


In [ ]:
import os
import glob
from google.colab import files

# 🔧 CHANGE THESE IF NEEDED
TASK = "acdc"   # or "la"
EXP_NAME = "acdc_colab_400epochs-task_acdc-ratio_0.1"

RCPS_ROOT = "/content/RCPS_ACDC"
CKPT_DIR  = os.path.join(RCPS_ROOT, "experiments", "checkpoints", TASK, EXP_NAME)

print("Checkpoint dir:", CKPT_DIR)
!ls "$CKPT_DIR"

# Find .pt files
pt_files = glob.glob(os.path.join(CKPT_DIR, "*.pt"))
print("\nFound .pt files:", [os.path.basename(f) for f in pt_files])

if not pt_files:
    print("\n❌ No .pt files found to zip. Check your checkpoint directory or training run.")
else:
    zip_name = f"{TASK}_{EXP_NAME}_weights.zip"
    zip_path = os.path.join(CKPT_DIR, zip_name)

    # IMPORTANT: no quotes around *.pt so the shell expands it
    !cd "$CKPT_DIR" && zip -r {zip_name} *.pt

    print("\nCreated zip at:", zip_path)
    files.download(zip_path)


In [ ]:
import os

TASK = "acdc"
EXP_NAME = "acdc_colab_400epochs-task_acdc-ratio_0.1"

RCPS_ROOT = "/content/RCPS_ACDC"
CKPT_DIR  = os.path.join(RCPS_ROOT, "experiments", "checkpoints", TASK, EXP_NAME)

print("Checkpoint directory:", CKPT_DIR)
!ls -l "$CKPT_DIR"


In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

# === 1. Set your paths (you already printed these above) ===
IMAGE_DIR = "/content/CML/data/ACDC_rcps/val_images"
LABEL_DIR = "/content/CML/data/ACDC_rcps/val_labels"
PRED_DIR  = "/content/RCPS_ACDC/experiments/inference_display/acdc/acdc_colab_400epochs-task_acdc-ratio_0.1"

print("IMAGE_DIR:", IMAGE_DIR)
print("LABEL_DIR:", LABEL_DIR)
print("PRED_DIR :", PRED_DIR)

# === 2. Define metric functions ===
def dice_score(pred, gt, eps=1e-8):
    inter = np.logical_and(pred, gt).sum()
    return (2 * inter + eps) / (pred.sum() + gt.sum() + eps)

def iou_score(pred, gt, eps=1e-8):
    inter = np.logical_and(pred, gt).sum()
    union = pred.sum() + gt.sum() - inter
    return (inter + eps) / (union + eps)

def precision_score(pred, gt, eps=1e-8):
    tp = np.logical_and(pred == 1, gt == 1).sum()
    fp = np.logical_and(pred == 1, gt == 0).sum()
    return (tp + eps) / (tp + fp + eps)

def recall_score(pred, gt, eps=1e-8):
    tp = np.logical_and(pred == 1, gt == 1).sum()
    fn = np.logical_and(pred == 0, gt == 1).sum()
    return (tp + eps) / (tp + fn + eps)

# === 3. Loop over prediction files and match labels ===
rows = []

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
print(f"\nFound {len(pred_files)} prediction files.")

for pred_fname in pred_files:
    # Example: "patient002_frame01_pred.nii.gz" -> "patient002_frame01.nii.gz"
    case_core = pred_fname.replace("_pred", "")
    label_fname = case_core                     # adjust here if your labels are named differently

    pred_path  = os.path.join(PRED_DIR, pred_fname)
    label_path = os.path.join(LABEL_DIR, label_fname)

    if not os.path.exists(label_path):
        print(f"⚠️  No label found for {pred_fname}, expected: {label_path}")
        continue

    # Load images
    pred_img = nib.load(pred_path).get_fdata()
    gt_img   = nib.load(label_path).get_fdata()

    # Treat any non-zero as foreground (binary mask)
    pred_fg = (pred_img > 0).astype(np.uint8)
    gt_fg   = (gt_img > 0).astype(np.uint8)

    d  = dice_score(pred_fg, gt_fg)
    j  = iou_score(pred_fg, gt_fg)
    p  = precision_score(pred_fg, gt_fg)
    r  = recall_score(pred_fg, gt_fg)

    rows.append({
        "case": case_core,
        "dice": d,
        "iou": j,
        "precision": p,
        "recall": r,
    })

# === 4. Put into a DataFrame and show summary ===
if len(rows) == 0:
    print("\nNo metrics computed — check that label filenames match the logic in the loop.")
else:
    df = pd.DataFrame(rows)
    display(df)

    print("\n===== ACDC AVERAGE METRICS =====")
    print("Mean Dice     :", df['dice'].mean())
    print("Mean IoU      :", df['iou'].mean())
    print("Mean Precision:", df['precision'].mean())
    print("Mean Recall   :", df['recall'].mean())
